**<span style="font-size:18px; color: LightBlue;">Creating file grouped by 1-hour sessions, with all item_id/value pairs in a single column</span>**

**<span style="font-size:14px; color: LightBlue;">Approach 1: with item_id numbered sequentially within each session</span>**

In [ ]:
# Creating file grouped by 1-hour sessions, with all item_id/value pairs in a single column(with item_id numbered sequentially within each session)
import pandas as pd

df = pd.read_csv("csv files/bigfile_part3.csv")

df["MONITORTIME"] = pd.to_datetime(df["MONITORTIME"])

df = df.sort_values(
    ["SUBJECT_ID", "HADM_ID", "VISIT_ID", "MONITORTIME"]
).reset_index(drop=True)

def add_2h_session_id(g: pd.DataFrame) -> pd.DataFrame:
    t = g["MONITORTIME"].to_numpy()
    session = []
    start = t[0]
    sid = 0
    for ts in t:
        if (ts - start) > pd.Timedelta(hours=1):
            sid += 1
            start = ts
        session.append(sid)
    g = g.copy()
    g["session_id"] = session
    return g

df = df.groupby(
    ["SUBJECT_ID", "HADM_ID", "VISIT_ID"],
    group_keys=False
).apply(add_2h_session_id)

def format_itemid_value(group: pd.DataFrame) -> str:
    pairs = []
    for i, (_, row) in enumerate(group.iterrows(), start=1):
        pairs.append(
            f"item_id{i}={row['ITEMID']}, value{i}={row['VALUE']}"
        )
    return "; ".join(pairs)

out = (
    df.groupby(
        ["SUBJECT_ID", "HADM_ID", "VISIT_ID", "session_id"],
        as_index=False
    )
    .agg(
        MONITORTIME=("MONITORTIME", "min"),
        ITEMID_VALUE=("ITEMID", lambda _: format_itemid_value(
            df.loc[_.index]
        ))
    )
)

out = out.drop(columns=["session_id"])

out.to_csv("csv files/aggregated_sessions3.csv", index=False)

print(out.head())
print(out.shape)


/var/folders/6c/h_2qq_495412tl4y7hpdmk5c0000gn/T/ipykernel_31112/3999456378.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["MONITORTIME"] = pd.to_datetime(df["MONITORTIME"])
/var/folders/6c/h_2qq_495412tl4y7hpdmk5c0000gn/T/ipykernel_31112/3999456378.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(add_2h_session_id)


   SUBJECT_ID  HADM_ID  VISIT_ID         MONITORTIME  \
0       11733   112733         1 1999-07-21 17:45:00   
1       11733   112733         2 1999-09-01 10:40:00   
2       11733   112733         2 1999-09-01 11:45:00   
3       11734   112121         1 2070-07-16 11:10:00   
4       11734   112121         1 2070-07-16 12:15:00   

                                        ITEMID_VALUE  
0  item_id1=SV7, value1=100; item_id2=SV2, value2...  
1  item_id1=SV1, value1=193; item_id2=SV7, value2...  
2  item_id1=SV4, value1=79; item_id2=SV3, value2=...  
3  item_id1=SV1, value1=99; item_id2=SV6, value2=...  
4  item_id1=SV7, value1=100; item_id2=SV6, value2...  
(3687, 5)


**<span style="font-size:14px; color: LightBlue;">Approach 2: with all itemid without numbering</span>**

In [ ]:
# Creating file grouped by 1-hour sessions, with item_id/value pairs in separate columns
import pandas as pd

df = pd.read_csv("bigfile_part1.csv")

df["MONITORTIME"] = pd.to_datetime(df["MONITORTIME"], errors="coerce")

df = df.sort_values(["SUBJECT_ID", "HADM_ID", "VISIT_ID", "MONITORTIME"]).reset_index(drop=True)

def add_2h_session_id(g: pd.DataFrame) -> pd.DataFrame:
    t = g["MONITORTIME"].to_numpy()
    session = []
    start = t[0]
    sid = 0
    for ts in t:
        if (ts - start) > pd.Timedelta(hours=2):
            sid += 1
            start = ts
        session.append(sid)
    g = g.copy()
    g["session_id"] = session
    return g

df = df.groupby(["SUBJECT_ID", "HADM_ID", "VISIT_ID"], group_keys=False).apply(add_2h_session_id)

agg = (
    df.groupby(["SUBJECT_ID", "HADM_ID", "VISIT_ID", "session_id"], as_index=False)
      .agg(
          MONITORTIME=("MONITORTIME", "min"),
          ITEMID_LIST=("ITEMID", list),
          VALUE_LIST=("VALUE", list),
      )
)

max_pairs = agg["ITEMID_LIST"].str.len().max()

for i in range(max_pairs):
    agg[f"ITEMID_{i+1}"] = agg["ITEMID_LIST"].apply(lambda x: x[i] if i < len(x) else pd.NA)
    agg[f"VALUE_{i+1}"]  = agg["VALUE_LIST"].apply(lambda x: x[i] if i < len(x) else pd.NA)

out = agg.drop(columns=["session_id", "ITEMID_LIST", "VALUE_LIST"])

print(out.head())
print(out.shape)


**<span style="font-size:18px; color: LightBlue;">Performing stats (min, max, mean, median) on itemID value (SV1-SV7)</span>**

In [ ]:
# Extracting SV1-SV7 stats from the file created above (with item_id/value pairs in a single column)
import re
import numpy as np
import pandas as pd

# ---------- CONFIG ----------
INPUT_CSV = "csv files/aggregated_sessions1.csv"
OUTPUT_XLSX = "xlsx files/output_with_sv_stats1.xlsx"

ITEM_IDS = [f"SV{i}" for i in range(1, 8)]
STATS = ["min", "max", "mean", "median"]  # average = mean (kept as separate col per request)

# Matches patterns like: item_id1=SV1, value1=73  (spaces optional)
PAIR_RE = re.compile(r"item_id\d+\s*=\s*(SV[1-7])\s*,\s*value\d+\s*=\s*([-+]?\d*\.?\d+)")


def parse_itemid_value(cell: str) -> dict[str, list[float]]:
    """Return dict like {'SV1':[...], 'SV2':[...], ...} for one row."""
    sv_map = {sv: [] for sv in ITEM_IDS}
    if pd.isna(cell):
        return sv_map

    text = str(cell)
    for sv, val in PAIR_RE.findall(text):
        try:
            sv_map[sv].append(float(val))
        except ValueError:
            # skip non-numeric values safely
            pass
    return sv_map


def compute_stats(values: list[float]) -> dict[str, float]:
    """Compute required stats for a list; return NaN if empty."""
    if not values:
        return {s: np.nan for s in STATS}

    arr = np.array(values, dtype=float)
    m = float(np.mean(arr))
    return {
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "mean": m,
        "median": float(np.median(arr)),
    }


# ---------- LOAD ----------
df = pd.read_csv(INPUT_CSV)

# ---------- BUILD SV STATS COLUMNS ----------
out_cols = {}

for idx, cell in df["ITEMID_VALUE"].items():
    sv_map = parse_itemid_value(cell)

    row_stats = {}
    for sv in ITEM_IDS:
        stats = compute_stats(sv_map[sv])
        for stat_name, stat_val in stats.items():
            row_stats[f"{sv}_{stat_name}"] = stat_val

    for k, v in row_stats.items():
        out_cols.setdefault(k, []).append(v)

stats_df = pd.DataFrame(out_cols)

# Keep original 4 columns + new stats columns
result = pd.concat([df[["SUBJECT_ID", "HADM_ID", "VISIT_ID", "MONITORTIME"]], stats_df], axis=1)

# ---------- SAVE ----------
# Requires: pip install openpyxl
result.to_excel(OUTPUT_XLSX, index=False)

print(f"Saved: {OUTPUT_XLSX}")


Saved: xlsx files/output_with_sv_stats1.xlsx
